# 01. AR·Diffusion·Block Diffusion Attention

**학습 목표**: 세 decoding 방식에서 response token이 볼 수 있는 위치를 Boolean mask로 구성합니다. 실제 GPU kernel이 아니라 정보 흐름과 정답 누출 조건을 이해하는 toy 실습입니다.

**실행 방법**: Python 3과 Jupyter에서 셀을 위에서 아래로 실행합니다. 외부 패키지는 필요하지 않습니다.

In [ ]:
def attention_mask(length, mode, block_size=4):
    mask = []
    for query in range(length):
        row = []
        for key in range(length):
            if mode == 'ar':
                allowed = key <= query
            elif mode == 'diffusion':
                allowed = True
            elif mode == 'block':
                # 이전 block 전체와 현재 block 내부를 볼 수 있습니다.
                query_block = query // block_size
                key_block = key // block_size
                allowed = key_block <= query_block
            else:
                raise ValueError(f'알 수 없는 mode: {mode}')
            row.append(allowed)
        mask.append(row)
    return mask

def show(mask):
    for row in mask:
        print(' '.join('●' if value else '·' for value in row))


In [ ]:
for mode in ('ar', 'diffusion', 'block'):
    print(f'\n[{mode}]')
    show(attention_mask(8, mode, block_size=4))


## 불변조건 확인

AR은 미래 위치를 보지 않습니다. Block diffusion의 첫 block은 내부 4칸을 모두 보지만 두 번째 block은 첫 block 전체도 볼 수 있습니다.

In [ ]:
ar = attention_mask(8, 'ar')
full = attention_mask(8, 'diffusion')
block = attention_mask(8, 'block', block_size=4)
assert not ar[2][3] and ar[3][2]
assert all(all(row) for row in full)
assert block[0][3] and not block[0][4]
assert block[4][0] and block[4][7]
print('모든 attention 불변조건을 통과했습니다.')


## 확장 과제

Image·prompt prefix는 모든 response 위치에서 보이도록 별도 열을 추가하세요. Corrupted stream이 같은 위치의 clean answer를 보지 못하도록 cross-stream mask도 구현해 보세요.